# LA Studio Speech-to-Text GPU Worker

Select **Runtime > Change runtime type > GPU**, then **Run all**. This starts a temporary Faster-Whisper CUDA worker for LA Studio Speech-to-Text. It is a direct Colab route: it does not read, send, proxy, or depend on API Gateway credentials. Copy the URL and temporary token printed in the final cell into the **Direct Colab GPU** settings in Speech-to-Text Studio.

In [ ]:
import subprocess, sys

def run(*args):
    print('+', ' '.join(map(str, args)))
    subprocess.run(args, check=True)

run('nvidia-smi')
run(sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'faster-whisper==1.1.1', 'fastapi>=0.115.0', 'uvicorn[standard]>=0.34.0', 'python-multipart>=0.0.20')


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_speech_worker.py')
WORKER.write_text(r'''
import os
import secrets
import shutil
import threading
from pathlib import Path

import ctranslate2
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from faster_whisper import WhisperModel

if ctranslate2.get_cuda_device_count() < 1:
    raise RuntimeError('CUDA is unavailable. Stop and select a GPU runtime; CPU fallback is disabled.')

TOKEN = os.environ['LA_STUDIO_COLAB_TOKEN']
MODEL_ID = os.environ.get('LA_STUDIO_STT_MODEL', 'large-v3')
MODEL_CACHE = os.environ.get('LA_STUDIO_STT_MODEL_CACHE', '/content/la-studio-stt-models')
UPLOADS = Path('/content/la-studio-stt-uploads')
UPLOADS.mkdir(parents=True, exist_ok=True)
MODEL_LOCK = threading.Lock()
# Bound untrusted tunnel traffic before it can consume Colab disk or GPU.
MAX_UPLOAD_BYTES = 512 * 1024 * 1024
MAX_AUDIO_SECONDS = 30 * 60
ALLOWED_CONTENT_TYPES = {'audio/wav', 'audio/x-wav', 'audio/mpeg', 'audio/mp4', 'video/mp4', 'audio/webm', 'video/webm', 'audio/ogg', 'audio/flac'}
REQUEST_SLOTS = threading.BoundedSemaphore(1)
MODEL = WhisperModel(MODEL_ID, device='cuda', compute_type='float16', download_root=MODEL_CACHE)
app = FastAPI(title='LA Studio Speech GPU Worker', docs_url=None, redoc_url=None, openapi_url=None)

def authorize(authorization: str | None) -> None:
    if authorization != 'Bearer ' + TOKEN:
        raise HTTPException(status_code=401, detail='invalid worker token')

@app.get('/health')
def health(authorization: str | None = Header(default=None)):
    authorize(authorization)
    return {'ready': True, 'device': 'cuda', 'model': MODEL_ID}

@app.get('/v1/health')
def v1_health(authorization: str | None = Header(default=None)):
    authorize(authorization)
    return {'ready': True, 'device': 'cuda', 'model': MODEL_ID, 'cpu_fallback': False}

@app.get('/v1/capabilities')
def capabilities(authorization: str | None = Header(default=None)):
    authorize(authorization)
    return {'contract_version': 1, 'device': 'cuda', 'capabilities': [{'id': 'stt', 'models': [{'id': 'faster-whisper-' + MODEL_ID, 'upstream_model': 'Systran/faster-whisper-' + MODEL_ID, 'languages': ['auto'], 'formats': ['verbose_json'], 'max_audio_seconds': MAX_AUDIO_SECONDS, 'device': 'cuda', 'loaded': True}]}]}

@app.post('/v1/audio/transcriptions')
async def transcribe(file: UploadFile = File(...), model: str = Form(default='colab'), response_format: str = Form(default='verbose_json'), language: str | None = Form(default=None), authorization: str | None = Header(default=None)):
    authorize(authorization)
    if response_format not in ('verbose_json', 'json'):
        raise HTTPException(status_code=400, detail='response_format must be verbose_json or json')
    if file.content_type not in ALLOWED_CONTENT_TYPES:
        raise HTTPException(status_code=415, detail='unsupported audio MIME type')
    suffix = Path(file.filename or 'audio.wav').suffix.lower() or '.wav'
    if suffix not in {'.wav', '.mp3', '.m4a', '.mp4', '.webm', '.ogg', '.flac'}:
        raise HTTPException(status_code=415, detail='unsupported audio filename extension')
    if not REQUEST_SLOTS.acquire(blocking=False):
        raise HTTPException(status_code=429, detail='the Colab STT worker is busy; retry shortly')
    upload = UPLOADS / (secrets.token_urlsafe(16) + suffix)
    try:
        with upload.open('wb') as output:
            while chunk := await file.read(1024 * 1024):
                output.write(chunk)
                if output.tell() > MAX_UPLOAD_BYTES:
                    raise HTTPException(status_code=413, detail='audio exceeds the 512 MB upload limit')
        if upload.stat().st_size <= 0:
            raise HTTPException(status_code=413, detail='audio must not be empty')
        language_code = None if not language or language.lower() == 'auto' else language
        with MODEL_LOCK:
            iterator, info = MODEL.transcribe(str(upload), language=language_code, beam_size=5, word_timestamps=True, vad_filter=True, condition_on_previous_text=True)
            segments = []
            words = []
            for index, segment in enumerate(iterator):
                text = segment.text.strip()
                if text:
                    segments.append({'id': index, 'start': segment.start, 'end': segment.end, 'text': text})
                for word in segment.words or []:
                    words.append({'word': word.word, 'start': word.start, 'end': word.end})
        if float(info.duration or 0.0) > MAX_AUDIO_SECONDS:
            raise HTTPException(status_code=413, detail='audio exceeds the 30 minute duration limit')
        text = ' '.join(item['text'] for item in segments)
        if not text:
            raise HTTPException(status_code=422, detail='the selected model produced an empty transcript')
        return {'text': text, 'language': info.language, 'duration': info.duration, 'segments': segments, 'words': words}
    finally:
        await file.close()
        upload.unlink(missing_ok=True)
        REQUEST_SLOTS.release()
''', encoding='utf-8')
print('Worker source prepared:', WORKER)


In [ ]:
import json, os, re, secrets, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env.update({'LA_STUDIO_COLAB_TOKEN': TOKEN, 'LA_STUDIO_STT_MODEL': 'large-v3', 'LA_STUDIO_STT_MODEL_CACHE': '/content/la-studio-stt-models'})
LOG = '/content/la-studio-speech-worker.log'
worker = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'la_studio_speech_worker:app', '--host', '127.0.0.1', '--port', '3921'], cwd='/content', env=env, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
for _ in range(150):
    try:
        request = urllib.request.Request('http://127.0.0.1:3921/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=5) as response:
            health = json.load(response)
        if health.get('ready') and health.get('device') == 'cuda':
            break
    except Exception:
        time.sleep(2)
else:
    worker.terminate()
    raise RuntimeError('Speech worker did not become CUDA-ready. Log tail:\n' + Path(LOG).read_text(errors='replace')[-4000:])
subprocess.run(['bash', '-lc', 'wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb'], check=True)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3921', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[^\s]+trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate(); tunnel.terminate()
    raise RuntimeError('Cloudflare tunnel URL was not found')
print('\nLA_STUDIO_COLAB_STT_URL=' + public_url)
print('LA_STUDIO_COLAB_STT_TOKEN=' + TOKEN)
print('MODEL=faster-whisper-large-v3; DEVICE=cuda')
print('Copy the URL and token into LA Studio. Do not add /v1 to the URL.')


In [ ]:
# ==============================================================================
# 📥 LƯU FILE TRỰC TIẾP VÀO THƯ MỤC DỰ ÁN TRÊN MÁY TÍNH (FILE SYSTEM ACCESS API)
# ==============================================================================
import base64
import glob
import json
import os
from IPython.display import HTML, display

# Thu thập tất cả các file kết quả vừa tạo
result_files = {}
for pattern in ['/content/*.wav', '/content/*.srt', '/content/*.json', '/content/*/*/*.wav', '/content/*/*/*.srt']:
    for f in glob.glob(pattern):
        name = os.path.basename(f)
        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:
            with open(f, 'rb') as fp:
                result_files[name] = base64.b64encode(fp.read()).decode('utf-8')

if not result_files:
    print("⚠️ Chưa có file kết quả mới để lưu.")
else:
    print(f"✅ Đã tìm thấy {len(result_files)} file kết quả: {', '.join(result_files.keys())}")
    print("👉 Bấm nút bên dưới và chọn thư mục 'LA-Studio/out/colab-live' để lưu thẳng vào máy:")
    
    files_json = json.dumps(result_files)
    html_code = f"""
    <button id="saveBtn" style="background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;">
        📁 Chọn Thư Mục & Lưu File Trực Tiếp Vào Máy
    </button>
    <div id="statusLog" style="margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;"></div>
    <script>
    document.getElementById('saveBtn').onclick = async () => {{
        const log = document.getElementById('statusLog');
        try {{
            if (!window.showDirectoryPicker) {{
                log.innerText = 'Trình duyệt không hỗ trợ File System Access API. Đang dùng tải thông thường...';
                return;
            }}
            log.innerText = 'Đang mở hộp thoại chọn thư mục...';
            const dirHandle = await window.showDirectoryPicker();
            const files = {files_json};
            for (const [name, b64] of Object.entries(files)) {{
                log.innerText = 'Đang ghi file: ' + name + '...';
                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});
                const writable = await fileHandle.createWritable();
                const byteCharacters = atob(b64);
                const byteNumbers = new Array(byteCharacters.length);
                for (let i = 0; i < byteCharacters.length; i++) {{
                    byteNumbers[i] = byteCharacters.charCodeAt(i);
                }}
                const byteArray = new Uint8Array(byteNumbers);
                await writable.write(byteArray);
                await writable.close();
            }}
            log.innerText = '🎉 Đã lưu thành công toàn bộ file vào thư mục bạn chọn!';
        }} catch (err) {{
            if (err.name !== 'AbortError') {{
                log.innerText = 'Lỗi: ' + err.message;
            }} else {{
                log.innerText = 'Đã hủy chọn thư mục.';
            }}
        }}
    }};
    </script>
    """
    display(HTML(html_code))
